# 01d - Replace Shrubland Test Cities and Expand Test Groups
Fixes the Shrubland gap by replacing Cape Town and Christchurch, the two cities that carried Shrubland but could never be trained on it, with cities in the same countries that do not. Also expands both generalization test groups by one city each. For the South Africa slot, 13 candidate cities are checked. For the New Zealand slot, 11 candidate cities are checked. Whichever city in each group has the smallest trace of any class absent from training is the one actually used.

## Cell 1: Install packages and authenticate Earth Engine
Same setup as the earlier data collection notebooks, connecting to the same Earth Engine project so these four cities land in the same dataset.

In [1]:
!pip install earthengine-api geemap -q

import ee
ee.Authenticate()
ee.Initialize(project='lulc-sentinel1')

## Cell 2: Define the candidate city bounding boxes
25 South Africa candidates are compared for the slot that Cape Town used to fill, including 12 subtropical KwaZulu-Natal and Limpopo lowveld towns added specifically because shrubland is less common in those regions than in the Cape/interior cities already ruled out. 11 New Zealand candidates are compared for the Christchurch slot, 5 of which already came back at literal 0.00%. This is a wide sweep since the land-mix check itself is a quick Earth Engine query, not an export, so checking many cities costs almost no time. Seoul and Bangkok are the two new cities for Option 2, both already confirmed clean.

In [2]:
def box(lon, lat, half_w=0.1, half_h=0.055):
    return ee.Geometry.Rectangle([lon - half_w, lat - half_h, lon + half_w, lat + half_h])

south_africa_centers = {
    "Johannesburg": (28.0473, -26.2041),
    "Durban": (31.0218, -29.8587),
    "Pretoria": (28.2293, -25.7479),
    "PortElizabeth": (25.6022, -33.9608),
    "EastLondon": (27.8546, -33.0292),
    "Bloemfontein": (26.1596, -29.0852),
    "Pietermaritzburg": (30.3794, -29.6006),
    "Kimberley": (24.7499, -28.7282),
    "Polokwane": (29.4689, -23.9045),
    "Nelspruit": (30.9694, -25.4753),
    "George": (22.4617, -33.9630),
    "Rustenburg": (27.2424, -25.6672),
    "Welkom": (26.7342, -27.9811),
    # Added: subtropical KwaZulu-Natal coast and Limpopo lowveld,
    # regions structurally less likely to carry shrubland than the Cape/interior cities above
    "RichardsBay": (32.0377, -28.7830),
    "Empangeni": (31.8930, -28.7520),
    "PortShepstone": (30.4547, -30.7418),
    "Ladysmith": (29.7814, -28.5600),
    "Newcastle": (29.9307, -27.7580),
    "Tzaneen": (30.1636, -23.8333),
    "Thohoyandou": (30.4845, -22.9458),
    "Giyani": (30.7187, -23.3012),
    "Kroonstad": (27.2347, -27.6539),
    "Klerksdorp": (26.6660, -26.8520),
    "Mahikeng": (25.6403, -25.8560),
    "Vryheid": (30.7913, -27.7698),
}

new_zealand_centers = {
    "Auckland": (174.7633, -36.8485),
    "Wellington": (174.7756, -41.2866),
    "Hamilton": (175.2793, -37.7870),
    "Tauranga": (176.1651, -37.6878),
    "Dunedin": (170.5028, -45.8788),
    "PalmerstonNorth": (175.6082, -40.3523),
    "Napier": (176.9120, -39.4928),
    "Nelson": (173.2840, -41.2706),
    "Rotorua": (176.2497, -38.1368),
    "NewPlymouth": (174.0752, -39.0556),
    "Invercargill": (168.3538, -46.4132),
}

# Limpopo lowveld towns have a subtropical/tropical savanna climate, genuinely different
# from the training set's temperate European cities. Kept in the comparison for reference,
# but excluded from the actual pick, since using one would quietly turn the
# "climate-similar-but-unseen" test group into something closer to the dissimilar-region group.
climate_excluded_sa = {"Tzaneen", "Thohoyandou", "Giyani"}

candidate_cities = {}
for name, (lon, lat) in south_africa_centers.items():
    candidate_cities[name] = box(lon, lat)
for name, (lon, lat) in new_zealand_centers.items():
    candidate_cities[name] = box(lon, lat)

# Already-confirmed clean, kept as fixed picks, not part of the ranking
candidate_cities["Seoul"] = ee.Geometry.Rectangle([126.8780, 37.5115, 127.0780, 37.6215])
candidate_cities["Bangkok"] = ee.Geometry.Rectangle([100.4018, 13.7013, 100.6018, 13.8113])

print(f"Total candidates: {len(candidate_cities)}")
print(f"South Africa candidates: {len(south_africa_centers)}")
print(f"New Zealand candidates: {len(new_zealand_centers)}")

Total candidates: 38
South Africa candidates: 25
New Zealand candidates: 11


## Cell 3: Verify land cover mix for all candidates
Pulls real ESA WorldCover class percentages for all South Africa and New Zealand candidates, plus Seoul and Bangkok. Checks every class absent from training, not just Shrubland. Ranks both country groups from lowest trace to highest. The South Africa ranking is shown two ways: the full list for reference, and an eligible-only list that excludes the Limpopo lowveld towns, since those don't match the training climate even if their Shrubland numbers look good. The eligible-only ranking is what actually gets used to pick a city.

In [3]:
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()

class_names = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare/sparse vegetation", 70: "Snow/ice",
    80: "Permanent water", 90: "Herbaceous wetland", 95: "Mangroves", 100: "Moss/lichen"
}

classes_absent_from_training = ["Shrubland", "Mangroves", "Snow/ice", "Moss/lichen"]

candidate_land_mix = {}
total_cities = len(candidate_cities)

for idx, (city, geom) in enumerate(candidate_cities.items()):
    hist = worldcover.clip(geom).reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=geom,
        scale=100,
        maxPixels=1e9
    ).getInfo()

    counts = hist['Map']
    total = sum(counts.values())
    pct_by_class = {}
    for code, count in counts.items():
        pct = (count / total) * 100
        name = class_names.get(int(float(code)), f"Unknown ({code})")
        pct_by_class[name] = round(pct, 2)

    candidate_land_mix[city] = pct_by_class
    print(f"[{idx+1}/{total_cities}] {city}: checked")

def untrained_trace(mix):
    return {cls: mix.get(cls, 0.0) for cls in classes_absent_from_training if mix.get(cls, 0.0) > 0}

print("\n--- South Africa candidates, full ranking (reference only, includes climate-excluded towns) ---")
sa_candidates = list(south_africa_centers.keys())
sa_ranked_all = sorted(sa_candidates, key=lambda c: sum(untrained_trace(candidate_land_mix[c]).values()))
for city in sa_ranked_all:
    total_trace = sum(untrained_trace(candidate_land_mix[city]).values())
    tag = " [climate-excluded, not eligible to be picked]" if city in climate_excluded_sa else ""
    print(f"{city}: total untrained-class trace = {total_trace:.2f}%  {untrained_trace(candidate_land_mix[city]) or '(clean)'}{tag}")

print("\n--- South Africa candidates, eligible-only ranking (this is what gets picked) ---")
sa_eligible = [c for c in sa_candidates if c not in climate_excluded_sa]
sa_ranked = sorted(sa_eligible, key=lambda c: sum(untrained_trace(candidate_land_mix[c]).values()))
for city in sa_ranked:
    total_trace = sum(untrained_trace(candidate_land_mix[city]).values())
    print(f"{city}: total untrained-class trace = {total_trace:.2f}%  {untrained_trace(candidate_land_mix[city]) or '(clean)'}")
print(f"Lowest eligible: {sa_ranked[0]}")

print("\n--- New Zealand candidates, ranked lowest trace first ---")
nz_candidates = list(new_zealand_centers.keys())
nz_ranked = sorted(nz_candidates, key=lambda c: sum(untrained_trace(candidate_land_mix[c]).values()))
for city in nz_ranked:
    total_trace = sum(untrained_trace(candidate_land_mix[city]).values())
    print(f"{city}: total untrained-class trace = {total_trace:.2f}%  {untrained_trace(candidate_land_mix[city]) or '(clean)'}")
print(f"Lowest: {nz_ranked[0]}")

[1/38] Johannesburg: checked
[2/38] Durban: checked
[3/38] Pretoria: checked
[4/38] PortElizabeth: checked
[5/38] EastLondon: checked
[6/38] Bloemfontein: checked
[7/38] Pietermaritzburg: checked
[8/38] Kimberley: checked
[9/38] Polokwane: checked
[10/38] Nelspruit: checked
[11/38] George: checked
[12/38] Rustenburg: checked
[13/38] Welkom: checked
[14/38] RichardsBay: checked
[15/38] Empangeni: checked
[16/38] PortShepstone: checked
[17/38] Ladysmith: checked
[18/38] Newcastle: checked
[19/38] Tzaneen: checked
[20/38] Thohoyandou: checked
[21/38] Giyani: checked
[22/38] Kroonstad: checked
[23/38] Klerksdorp: checked
[24/38] Mahikeng: checked
[25/38] Vryheid: checked
[26/38] Auckland: checked
[27/38] Wellington: checked
[28/38] Hamilton: checked
[29/38] Tauranga: checked
[30/38] Dunedin: checked
[31/38] PalmerstonNorth: checked
[32/38] Napier: checked
[33/38] Nelson: checked
[34/38] Rotorua: checked
[35/38] NewPlymouth: checked
[36/38] Invercargill: checked
[37/38] Seoul: checked
[38/3

## Cell 4: Lock in the final city choices and build the export list
South Africa is set directly to East London, not auto-picked by lowest trace. East London's untrained-class trace (0.44%) is slightly higher than Durban's (0.13%), but East London has a confirmed Cfb oceanic climate, the same classification shared by several training cities (London, Amsterdam, Brussels, and others), while Durban's Cfa humid subtropical climate does not match the training set as well. New Zealand is still auto-picked from the eligible ranking, since Hamilton won fairly at a genuine 0.00% trace with no climate trade-off involved.

In [4]:
# South Africa: set directly to East London. Not auto-picked by lowest trace,
# since Durban scores lower on Shrubland/Mangroves trace (0.13% vs 0.44%) but has
# the wrong climate type (Cfa humid subtropical) for this test group's purpose.
# East London is confirmed Cfb oceanic, matching several training cities' climate directly.
CHOSEN_SA_CITY = 'EastLondon'
assert CHOSEN_SA_CITY in candidate_cities, "EastLondon must be one of the checked candidates"

# New Zealand: auto-picked, Hamilton won fairly at a genuine 0.00% trace
CHOSEN_NZ_CITY = nz_ranked[0]

print(f"Chosen South Africa city: {CHOSEN_SA_CITY} (fixed choice, climate-matched, trace = "
      f"{sum(untrained_trace(candidate_land_mix[CHOSEN_SA_CITY]).values()):.2f}%)")
print(f"Chosen New Zealand city: {CHOSEN_NZ_CITY} (auto-picked lowest trace, "
      f"trace = {sum(untrained_trace(candidate_land_mix[CHOSEN_NZ_CITY]).values()):.2f}%)")

new_test_cities = {
    CHOSEN_SA_CITY: candidate_cities[CHOSEN_SA_CITY],
    CHOSEN_NZ_CITY: candidate_cities[CHOSEN_NZ_CITY],
    "Seoul": candidate_cities["Seoul"],
    "Bangkok": candidate_cities["Bangkok"],
}

print(f"\nFinal cities to export ({len(new_test_cities)}): {list(new_test_cities.keys())}")
print(f"Dropped (not exported): {[c for c in candidate_cities if c not in new_test_cities]}")

Chosen South Africa city: EastLondon (fixed choice, climate-matched, trace = 0.44%)
Chosen New Zealand city: Hamilton (auto-picked lowest trace, trace = 0.00%)

Final cities to export (4): ['EastLondon', 'Hamilton', 'Seoul', 'Bangkok']
Dropped (not exported): ['Johannesburg', 'Durban', 'Pretoria', 'PortElizabeth', 'Bloemfontein', 'Pietermaritzburg', 'Kimberley', 'Polokwane', 'Nelspruit', 'George', 'Rustenburg', 'Welkom', 'RichardsBay', 'Empangeni', 'PortShepstone', 'Ladysmith', 'Newcastle', 'Tzaneen', 'Thohoyandou', 'Giyani', 'Kroonstad', 'Klerksdorp', 'Mahikeng', 'Vryheid', 'Auckland', 'Wellington', 'Tauranga', 'Dunedin', 'PalmerstonNorth', 'Napier', 'Nelson', 'Rotorua', 'NewPlymouth', 'Invercargill']


## Cell 5: Mount Drive
Uses the same export folder as every other city in this project, so these four land in the same dataset.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_FOLDER = '/content/drive/MyDrive/lulc_dataset'
os.makedirs(DRIVE_FOLDER, exist_ok=True)
print("Export folder ready at:", DRIVE_FOLDER)

Mounted at /content/drive
Export folder ready at: /content/drive/MyDrive/lulc_dataset


## Cell 6: Define the Sentinel-1 + WorldCover export function
Identical logic to the earlier data collection notebooks: pulls one Sentinel-1 VV/VH monthly median composite and the WorldCover label for a city and month, and starts an export task for each to Google Drive.

In [6]:
import calendar

def get_month_range(year, month):
    start = f"{year}-{month:02d}-01"
    last_day = calendar.monthrange(year, month)[1]
    end = f"{year}-{month:02d}-{last_day:02d}"
    return start, end

def export_city_month(city_name, geom, year, month):
    start, end = get_month_range(year, month)

    s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(geom) \
        .filterDate(start, end) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW'))

    count = s1.size().getInfo()
    if count == 0:
        print(f"  [SKIP] {city_name} {year}-{month:02d}: no Sentinel-1 images found")
        return None

    s1_image = s1.median().select(['VV', 'VH']).clip(geom)
    worldcover_img = ee.ImageCollection('ESA/WorldCover/v200').first().clip(geom)

    tag = f"{city_name}_{year}_{month:02d}"

    s1_task = ee.batch.Export.image.toDrive(
        image=s1_image,
        description=f"s1_{tag}",
        folder='lulc_dataset',
        fileNamePrefix=f"s1_{tag}",
        region=geom,
        scale=10,
        maxPixels=1e9
    )
    s1_task.start()

    wc_task = ee.batch.Export.image.toDrive(
        image=worldcover_img,
        description=f"wc_{tag}",
        folder='lulc_dataset',
        fileNamePrefix=f"wc_{tag}",
        region=geom,
        scale=10,
        maxPixels=1e9
    )
    wc_task.start()

    print(f"  [EXPORT STARTED] {tag} ({count} Sentinel-1 images found)")
    return {
        "city": city_name,
        "year": year,
        "month": month,
        "s1_file": f"s1_{tag}.tif",
        "wc_file": f"wc_{tag}.tif",
        "s1_image_count": count,
    }

## Cell 7: Start exports for all four cities across all 12 months
Uses the same Nov 2025 to Oct 2026 window as every other city, so these four cities cover the same period as the rest of the dataset.

In [7]:
MONTHS = [
    (2025, 11), (2025, 12), (2026, 1), (2026, 2), (2026, 3), (2026, 4),
    (2026, 5), (2026, 6), (2026, 7), (2026, 8), (2026, 9), (2026, 10),
]

new_metadata_records = []
total_cities = len(new_test_cities)

for idx, (city_name, geom) in enumerate(new_test_cities.items()):
    print(f"\n--- [{idx+1}/{total_cities}] {city_name} ---")
    for year, month in MONTHS:
        record = export_city_month(city_name, geom, year, month)
        if record:
            new_metadata_records.append(record)

print(f"\nAll export tasks started. Total new city-month pairs queued: {len(new_metadata_records)}")


--- [1/4] EastLondon ---
  [EXPORT STARTED] EastLondon_2025_11 (3 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2025_12 (2 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2026_01 (2 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2026_02 (2 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2026_03 (3 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2026_04 (2 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2026_05 (1 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2026_06 (3 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2026_07 (3 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2026_08 (2 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2026_09 (3 Sentinel-1 images found)
  [EXPORT STARTED] EastLondon_2026_10 (2 Sentinel-1 images found)

--- [2/4] Hamilton ---
  [EXPORT STARTED] Hamilton_2025_11 (8 Sentinel-1 images found)
  [EXPORT STARTED] Hamilton_2025_12 (7 Sentinel-1 images found)
  [EXPORT STARTED] Hamilton_20

## Cell 8: Poll export tasks until all are finished
Earth Engine runs exports as background server-side tasks. This checks their status every 60 seconds and prints progress automatically until every task has completed or failed, so this cell does not need to be re-run manually.

In [8]:
import time

expected_descriptions = set()
for rec in new_metadata_records:
    tag = f"{rec['city']}_{rec['year']}_{rec['month']:02d}"
    expected_descriptions.add(f"s1_{tag}")
    expected_descriptions.add(f"wc_{tag}")

total_tasks = len(expected_descriptions)
print(f"Tracking {total_tasks} tasks by exact name match.")

check_interval_seconds = 60
check_number = 0

while True:
    check_number += 1
    all_tasks = ee.batch.Task.list()

    status_counts = {}
    seen_descriptions = set()
    for t in all_tasks:
        st = t.status()
        desc = st.get('description', '')
        if desc in expected_descriptions:
            seen_descriptions.add(desc)
            state = st['state']
            status_counts[state] = status_counts.get(state, 0) + 1

    completed = status_counts.get('COMPLETED', 0)
    failed = status_counts.get('FAILED', 0)
    running = status_counts.get('RUNNING', 0)
    ready = status_counts.get('READY', 0)
    missing = total_tasks - len(seen_descriptions)

    print(f"[Check {check_number}] completed={completed}/{total_tasks} "
          f"running={running} ready={ready} failed={failed} missing={missing}  "
          f"(status breakdown: {status_counts})")

    if completed + failed >= total_tasks:
        print("\nAll export tasks have finished.")
        if failed > 0:
            print(f"Warning: {failed} task(s) failed. Review the task list before continuing.")
        if missing > 0:
            print(f"Warning: {missing} expected task(s) were never found in the task list. Check names.")
        break

    time.sleep(check_interval_seconds)


Tracking 96 tasks by exact name match.
[Check 1] completed=43/96 running=2 ready=267 failed=0 missing=0  (status breakdown: {'READY': 267, 'RUNNING': 2, 'COMPLETED': 43})
[Check 2] completed=48/96 running=0 ready=264 failed=0 missing=0  (status breakdown: {'READY': 264, 'COMPLETED': 48})
[Check 3] completed=48/96 running=0 ready=264 failed=0 missing=0  (status breakdown: {'READY': 264, 'COMPLETED': 48})
[Check 4] completed=48/96 running=0 ready=264 failed=0 missing=0  (status breakdown: {'READY': 264, 'COMPLETED': 48})
[Check 5] completed=48/96 running=0 ready=264 failed=0 missing=0  (status breakdown: {'READY': 264, 'COMPLETED': 48})
[Check 6] completed=48/96 running=0 ready=264 failed=0 missing=0  (status breakdown: {'READY': 264, 'COMPLETED': 48})
[Check 7] completed=48/96 running=0 ready=264 failed=0 missing=0  (status breakdown: {'READY': 264, 'COMPLETED': 48})
[Check 8] completed=48/96 running=1 ready=263 failed=0 missing=0  (status breakdown: {'READY': 263, 'RUNNING': 1, 'COMPLE

## Cell 9: Merge new metadata with the existing dataset index
Appends Johannesburg, Auckland, Seoul, and Bangkok to `dataset_metadata.csv`, bringing the dataset to 32 cities. Safe to re-run: duplicate city-month rows are removed before saving. The save itself retries automatically on a Drive disconnect, remounting Drive before the final attempt, since a long run earlier in this project showed this step can silently fail otherwise.

In [9]:
import pandas as pd
import time

def with_retries(func, *args, retries=3, delay=15, **kwargs):
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except OSError as e:
            last_error = e
            print(f"  [retry {attempt}/{retries}] disk error: {e}")
            if attempt == retries:
                break
            if attempt == retries - 1:
                try:
                    print("  [retry] remounting Drive...")
                    drive.mount('/content/drive', force_remount=True)
                except Exception as mount_err:
                    print(f"  [retry] remount failed: {mount_err}")
            time.sleep(delay)
    raise last_error


existing_metadata_path = f"{DRIVE_FOLDER}/dataset_metadata.csv"
existing_df = pd.read_csv(existing_metadata_path)
new_df = pd.DataFrame(new_metadata_records)

combined_df = pd.concat([existing_df, new_df], ignore_index=True)

before_dedup = len(combined_df)
combined_df = combined_df.drop_duplicates(subset=['city', 'year', 'month'], keep='first').reset_index(drop=True)
after_dedup = len(combined_df)

with_retries(combined_df.to_csv, existing_metadata_path, index=False)

print(f"Existing records before this run: {len(existing_df)}")
print(f"New records added this run: {len(new_df)}")
print(f"Duplicates removed: {before_dedup - after_dedup}")
print(f"Final combined total: {after_dedup}")
print(f"Total cities in dataset: {combined_df['city'].nunique()}")
combined_df.groupby('city').size()

Existing records before this run: 336
New records added this run: 48
Duplicates removed: 0
Final combined total: 384
Total cities in dataset: 32


,0
city,
Amsterdam,12
Bangkok,12
Berlin,12
Bremen,12
Brussels,12
Cairo,12
CapeTown,12
Christchurch,12
Copenhagen,12


## Cell 10: Role reminder for the next preprocessing run
Prints the updated five-group city role assignment to copy into `02_preprocessing_28cities_v3.ipynb` Cell 8. Cape Town and Christchurch are dropped from `test_similar_unseen`, Johannesburg, Auckland, and Seoul replace and extend that group, and Bangkok is added to `test_dissimilar`. Training is unchanged.

In [10]:
train_cities = [
    'Kiel', 'Leipzig', 'Dresden', 'Hamburg', 'Berlin', 'Munich', 'Stuttgart', 'Munster',
    'Portland', 'Toronto', 'Vancouver', 'Melbourne', 'Vienna', 'Zurich',
    'London', 'Amsterdam', 'Brussels', 'Copenhagen', 'Warsaw', 'Prague',
]
val_cities = ['Freiburg']
test_same_region_cities = ['Rostock', 'Bremen']
test_similar_unseen_cities = [CHOSEN_SA_CITY, CHOSEN_NZ_CITY, 'Seoul']
test_dissimilar_cities = ['Mumbai', 'Lagos', 'Cairo', 'Bangkok']

print(f"train_cities ({len(train_cities)}):", train_cities)
print(f"val_cities ({len(val_cities)}):", val_cities)
print(f"test_same_region_cities ({len(test_same_region_cities)}):", test_same_region_cities)
print(f"test_similar_unseen_cities ({len(test_similar_unseen_cities)}):", test_similar_unseen_cities)
print(f"test_dissimilar_cities ({len(test_dissimilar_cities)}):", test_dissimilar_cities)
print(f"\nTotal cities across all roles: {len(train_cities) + len(val_cities) + len(test_same_region_cities) + len(test_similar_unseen_cities) + len(test_dissimilar_cities)}")
print("\nNote: Cape Town and Christchurch are no longer used anywhere in the dataset design.")

train_cities (20): ['Kiel', 'Leipzig', 'Dresden', 'Hamburg', 'Berlin', 'Munich', 'Stuttgart', 'Munster', 'Portland', 'Toronto', 'Vancouver', 'Melbourne', 'Vienna', 'Zurich', 'London', 'Amsterdam', 'Brussels', 'Copenhagen', 'Warsaw', 'Prague']
val_cities (1): ['Freiburg']
test_same_region_cities (2): ['Rostock', 'Bremen']
test_similar_unseen_cities (3): ['EastLondon', 'Hamilton', 'Seoul']
test_dissimilar_cities (4): ['Mumbai', 'Lagos', 'Cairo', 'Bangkok']

Total cities across all roles: 30

Note: Cape Town and Christchurch are no longer used anywhere in the dataset design.
